# 🐼 Manipulación de DataFrames con Pandas

**Objetivo:** Practicar el manejo de valores faltantes (NaN), el tratamiento de columnas categóricas y la fusión de DataFrames.

---

## Índice
1. Crear el DataFrame
2. Identificar valores faltantes
3. Rellenar columnas numéricas con la media
4. Rellenar columnas categóricas con la moda
5. Convertir `ciudad` a categórica
6. Agregar una nueva fila
7. Reconvertir `ciudad` a categórica
8. Verificar categorías actuales
9. Establecer y reordenar categorías
10. Crear segundo DataFrame y hacer merge


## 1. Crear el DataFrame

Creamos el DataFrame a partir de una lista de diccionarios. Algunos campos tienen `None` (valores faltantes) que trataremos en los siguientes pasos.

In [1]:
import pandas as pd

data = [
    {"nombre": "Juan", "edad": 25, "ciudad": "Madrid", "ingresos": 3000},
    {"nombre": "Ana", "edad": None, "ciudad": None, "ingresos": 2500},
    {"nombre": "Pedro", "edad": 30, "ciudad": "Barcelona", "ingresos": None},
    {"nombre": None, "edad": None, "ciudad": "Valencia", "ingresos": None},
    {"nombre": "Luisa", "edad": 35, "ciudad": "Madrid", "ingresos": None}
]

df = pd.DataFrame(data)

## 2. Identificar valores faltantes

Usamos `isna().sum()` para contar cuántos valores faltantes hay en cada columna.

In [2]:
print("Resultado para ver cuántos valores faltantes hay en cada columna:")
print(df.isna().sum())

Resultado para ver cuántos valores faltantes hay en cada columna:
nombre      1
edad        2
ciudad      1
ingresos    3
dtype: int64


## 3. Rellenar columnas numéricas con la media

Las columnas `edad` e `ingresos` son numéricas. Rellenamos sus NaN con la **media** de cada columna usando `fillna()`.

In [3]:
# Rellenar valores faltantes con la media de cada columna
df = df.fillna({'edad': df['edad'].mean(), 'ingresos': df['ingresos'].mean()})
print("\nDataFrame después de rellenar los valores faltantes con la media:")
print(df)


DataFrame después de rellenar los valores faltantes con la media:
  nombre  edad     ciudad  ingresos
0   Juan  25.0     Madrid    3000.0
1    Ana  30.0        NaN    2500.0
2  Pedro  30.0  Barcelona    2750.0
3    NaN  30.0   Valencia    2750.0
4  Luisa  35.0     Madrid    2750.0


## 4. Rellenar columnas categóricas con la moda

Las columnas `nombre` y `ciudad` son categóricas (texto). Rellenamos sus NaN con el valor más frecuente (**moda**) usando `mode()[0]`.

In [4]:
# Rellenar valores faltantes con la moda de cada columna
df = df.fillna({'nombre': df['nombre'].mode()[0], 'ciudad': df['ciudad'].mode()[0]})
print("\nDataFrame después de rellenar los valores faltantes con la moda:")
print(df)


DataFrame después de rellenar los valores faltantes con la moda:
  nombre  edad     ciudad  ingresos
0   Juan  25.0     Madrid    3000.0
1    Ana  30.0     Madrid    2500.0
2  Pedro  30.0  Barcelona    2750.0
3    Ana  30.0   Valencia    2750.0
4  Luisa  35.0     Madrid    2750.0


## 5. Convertir `ciudad` a variable categórica

Convertimos la columna `ciudad` al tipo `category` de Pandas, que es más eficiente en memoria y permite ordenar las categorías.

In [5]:
df['ciudad'] = df['ciudad'].astype('category')
print(df.dtypes)

nombre           str
edad         float64
ciudad      category
ingresos     float64
dtype: object


## 6. Agregar una nueva fila

Añadimos una nueva fila con `pd.concat()`. La nueva fila incluye la ciudad "Sevilla", que no existía como categoría.

In [6]:
new_row = pd.DataFrame([{"nombre": "Nuevo", "edad": 40, "ciudad": "Sevilla", "ingresos": 4000}])
df = pd.concat([df, new_row], axis="rows")
print("\nDataFrame después de agregar una nueva fila:")
print(df)


DataFrame después de agregar una nueva fila:
  nombre  edad     ciudad  ingresos
0   Juan  25.0     Madrid    3000.0
1    Ana  30.0     Madrid    2500.0
2  Pedro  30.0  Barcelona    2750.0
3    Ana  30.0   Valencia    2750.0
4  Luisa  35.0     Madrid    2750.0
0  Nuevo  40.0    Sevilla    4000.0


## 7. Reconvertir `ciudad` a categórica

Al hacer `concat()`, la columna `ciudad` pierde el tipo `category`. La reconvertimos para que incluya "Sevilla" como nueva categoría.

In [7]:
# Reconvertiendo 'ciudad' a categoría después de agregar la nueva fila
df['ciudad'] = df['ciudad'].astype('category')
print(df.dtypes)

nombre           str
edad         float64
ciudad      category
ingresos     float64
dtype: object


## 8. Verificar las categorías actuales

Comprobamos con `cat.categories` qué categorías reconoce Pandas en la columna `ciudad`.

In [8]:
print(df['ciudad'].cat.categories)

Index(['Barcelona', 'Madrid', 'Sevilla', 'Valencia'], dtype='str')


## 9. Establecer y reordenar categorías

Fijamos el conjunto exacto de categorías con `set_categories()` y las reordenamos con `reorder_categories()` para definir un orden explícito.

In [9]:
df['ciudad'] = df['ciudad'].cat.set_categories(["Madrid", "Barcelona", "Valencia", "Sevilla"])
df['ciudad'] = df['ciudad'].cat.reorder_categories(["Madrid", "Barcelona", "Valencia", "Sevilla"])

## 10. Crear segundo DataFrame y hacer merge

Creamos un segundo DataFrame con la profesión de cada persona y lo fusionamos con el original usando `merge()` sobre la columna `nombre`.

- `how="inner"` → solo filas con coincidencia en ambos DataFrames.
- `validate="many_to_one"` → permite que `nombre` se repita en el DataFrame izquierdo, pero no en el derecho.

In [10]:
data_extra = [
    {"nombre": "Juan", "profesion": "Ingeniero"},
    {"nombre": "Ana", "profesion": "Médico"},
    {"nombre": "Pedro", "profesion": "Abogado"},
    {"nombre": "Luisa", "profesion": "Diseñadora"},
    {"nombre": "Nuevo", "profesion": "Artista"}
]

result = pd.merge(df, pd.DataFrame(data_extra), on="nombre", how="inner", validate="many_to_one")
print("\nDataFrame después de hacer el merge:")
print(result)


DataFrame después de hacer el merge:
  nombre  edad     ciudad  ingresos   profesion
0   Juan  25.0     Madrid    3000.0   Ingeniero
1    Ana  30.0     Madrid    2500.0      Médico
2  Pedro  30.0  Barcelona    2750.0     Abogado
3    Ana  30.0   Valencia    2750.0      Médico
4  Luisa  35.0     Madrid    2750.0  Diseñadora
5  Nuevo  40.0    Sevilla    4000.0     Artista
